# 12 · Background commands & snapshots (MicroVM)

How to launch long-running background processes, stream and
control them, expose a network port, and persist a sandbox's files across sessions.

**Key concepts**
- **Background commands** — `commands.run(..., background=True)` returns a
  `CommandHandle` you can iterate (stream), `wait()` on, `kill()`, or feed stdin —
  for builds, servers, test runs that outlive a single call.
- **`get_host(port)`** — MicroVMs expose a real inbound HTTPS endpoint, so a server
  inside the VM is reachable from outside. (AgentCore backends have no ingress.)
- **pause / resume** — `pause()` snapshots the workspace **files** to S3 and stops
  the VM; `resume()` restores them into a fresh VM. *Files survive; in-memory state
  does not* — a pragmatic "durable sandbox".
- These features need the **`microvm`** backend (a full OS VM), not `agentcore-ci`.

**Prerequisites:** a built MicroVM image ARN (`RAITBOX_MICROVM_IMAGE`) and, for
pause/resume, an S3 bucket (`RAITBOX_SNAPSHOT_BUCKET`). See `docs/INTEGRATION.md`.

In [ ]:
%pip install -q raitbox

### Configuration
The `microvm` backend needs a built image ARN — read it from the environment.

In [ ]:
import os

from raitbox import Sandbox

IMAGE_ARN = os.environ["RAITBOX_MICROVM_IMAGE"]

## Background command with streaming, plus list & kill

In [ ]:
with Sandbox.create(backend="microvm", template=IMAGE_ARN) as sbx:
    handle = sbx.commands.run(
        "for i in $(seq 1 5); do echo tick $i; sleep 1; done", background=True
    )
    print("pid:", handle.pid)
    for chunk in handle:          # stream output as it arrives
        print(chunk, end="")
    print("exit:", handle.wait().exit_code)

    other = sbx.commands.run("sleep 60", background=True)
    print("running:", [p.pid for p in sbx.commands.list()])
    other.kill()

    print("reachable at:", sbx.get_host(8080))

## Pause to S3, then resume into a fresh sandbox

In [ ]:
bucket = os.environ.get("RAITBOX_SNAPSHOT_BUCKET")

with Sandbox.create(backend="microvm", template=IMAGE_ARN) as sbx:
    sbx.files.write("keep.txt", "data")
    sandbox_id = sbx.pause(bucket=bucket)

restored = Sandbox.resume(sandbox_id, bucket=bucket, backend="microvm", template=IMAGE_ARN)
print("restored:", restored.files.read("keep.txt"))
restored.kill()

## Recap & where to go next
You ran and controlled background processes, reached a port via `get_host`, and
snapshotted files with pause/resume.

**Related:** `01-basics` (why microvm) · `14-swebench-lite` / `15-swe-gym-env`
(microvm-based coding evals).